# Apply Rolling Trimmed Mean along the Raster Dimension
Need to reshape the FDM to create a raster dimension so that the rolling trimmed mean can be applied to each raster individually. Unfortunately, there are missing images in a few rasters, meaning that it is not trivial to reshape the array.
This Jupyter notebook performs the following steps to the output of the `despike_and_save.ipynb` notebook:
    - Identify the missing images and insert temporary nan images in those locations
    - Reshape the array to create the raster dimension
    - Apply the rolling trimmed mean along each raster
    - Reshape to the original array shape
    - Remove the temporary inserted images
    - Pickle and save the array
    
This array will then be used in the `background_subtraction.ipynb` Jupyter notebook.

#### Import Statements

In [ ]:
%reload_ext autoreload
%autoreload 2
# %matplotlib notebook
%matplotlib inline

import pathlib as pl
from iris_mosaics import MosaicConfig, plan_rasters, read_pointing
import numpy as np
import pickle
import numba
import matplotlib.pyplot as plt
# params = {"ytick.color" : "k",
#           "xtick.color" : "k",
#           "axes.labelcolor" : "k",
#           "axes.edgecolor" : "k"}
# plt.rcParams.update(params)
from mpl_toolkits.axes_grid1 import ImageGrid
# import pandas as pd
import ndfilters
from matplotlib import colors
import astropy.units as u
from astropy import constants as const
from iris_mosaics import wcs_to_bins, spectral_plot, read_sg_image, read_sg_image_lvl1
import iris_mosaics as iris_fdm
from IPython.display import display, Math, Markdown
from astropy.visualization import quantity_support, time_support
from scipy.optimize import curve_fit, minimize
from astropy.modeling import models, fitting
from astropy.convolution import convolve, Gaussian1DKernel, Gaussian2DKernel
import astropy.time
quantity_support()
time_support()

## Load data

#### File paths of Level 1.1 FDM data and FUV background image

In [ ]:
cfg = MosaicConfig.load('20240811')   # <-- the only per-mosaic edit needed
path = cfg.data_root
# Path of Level 1.1 spectrograph image files (used to save the files at the end using their original names but in a new folder)
path_fdm = path / 'level_11_plus_iris_prep_bg_sub'
files = list(path_fdm.glob('*.fits'))

# Paths of pickled despiked Level 1.1 data
path_dspk_1394 = path / 'level_11_fpr_despiked_1394.pickle'
path_dspk_1403 = path / 'level_11_fpr_despiked_1403.pickle'

#### Select Si IV 1394 & 1403 regions
Manually choose area where data is -- WCS doesn't seem to work -- my guess is because the geometric correction hasn't been applied yet.

In [ ]:
# Number of images
num_imgs = len(files)

# Read in an image in order to see where to crop it down:
file_num = 1000
w_0, hdu_0, _ = read_sg_image(files[file_num],'fuv2')
img_0 = hdu_0[0].data

# Lengths of dimensions of total image
num_y_total = img_0.shape[0]
num_x_total = img_0.shape[1]

# Create masks for the areas of interest (Si IV 1394 & 1403).
# sl_1394 = slice(10,~9), slice(732,~215)
# sl_1403 = slice(10,~9), slice(868,~7)

# For the Aug 2024 mosaic that is twice as wide in the x direction...
x1 = 2 * 732
x2 = 2 * 215
x3 = 2 * 868
x4 = 2 * 8
sl_1394 = slice(10,~9), slice(x1,~x2)
sl_1403 = slice(10,~9), slice(x3,~x4)

# Si IV 1394 and 1403 image dimension lengths
num_y = img_0[sl_1394].shape[0]
num_x_1394 = img_0[sl_1394].shape[1]
num_x_1403 = img_0[sl_1403].shape[1]

# Treat the upper and lower parts of the CCD separately
# There is an interface where the two detector taps sit next to each other, causing a discontinuity in the image.

# Define the 1394 upper and lower slices
sl_1394_up = slice(0,264)
sl_1394_down = slice(264, None)

# Define the 1403 upper and lower slices
sl_1403_up = slice(0,264)
sl_1403_down = slice(264, None)

# New y-axis shape
num_y_split = img_0[sl_1394_up].shape[0]

#### Read in despiked Level 1.1 data

In [ ]:
with open(str(path_dspk_1394), 'rb') as f:
    sg_1394_dspk = pickle.load(f)

with open(str(path_dspk_1403), 'rb') as f:
    sg_1403_dspk = pickle.load(f)
    
# Create NaN mask
nan_mask_1394 = ~np.isfinite(sg_1394_dspk)
nan_mask_1403 = ~np.isfinite(sg_1403_dspk)

# Mean images of the despiked data
sg_1394_dspk_mean_image = np.nanmean(sg_1394_dspk, axis=0)
sg_1403_dspk_mean_image = np.nanmean(sg_1403_dspk, axis=0)

#### Define plot function to better display Si IV 1394 and 1403 together

In [ ]:
# plot_lines_sidebyside now lives in the package so every notebook shares one copy.
from iris_mosaics import plot_lines_sidebyside


### Separate images into 64-step rasters
We will apply a rolling median filter to each raster separately since there are harsh breaks between each of them

#### Identify and fill in missing images

Images go missing both inside rasters (a gap in `solar_x` larger than the
nominal 2-arcsec step) and off the ends of a raster. `plan_rasters` works out
where, without touching the data; `layout.pad` then inserts the NaN frames.

`layout.pad` and `layout.unpad` are exact inverses — `unpad` removes precisely
the frames `pad` inserted, tracked by index.


In [ ]:
%%time
# Pointing and observation time from each file's header
solar_x, solar_y, t_obs = read_pointing(files)


Plots of `diff(solar_x)` vs image # and `solar_x` and `solar_y` vs image #


In [ ]:
num_img_per_raster = cfg.num_img_per_raster

# Ideally where the breaks between rasters should be
index_raster = np.arange(0, solar_x.shape[0], num_img_per_raster)

plt.figure(figsize=(9,7))
plt.plot(np.diff(solar_x))
plt.ylim((-.5,35))
for i in index_raster:
    plt.axvline(i, linewidth=0.5, color='grey')
plt.ylabel('diff(solar_x) [arcsec]')
plt.xlabel('image #')

plt.figure(figsize=(9,7))
plt.scatter(solar_x, solar_y, c=np.arange(solar_x.shape[0]), s=4)
plt.colorbar(label='image #')
plt.xlabel('solar x [arcsec]')
plt.ylabel('solar y [arcsec]')


Work out the raster layout, then inspect what it found before padding anything.


In [ ]:
layout = plan_rasters(
    solar_x,
    num_img_per_raster=cfg.num_img_per_raster,
    step_arcsec=cfg.raster_step.value,
)

print(f'images in  : {layout.num_images_original}')
print(f'images out : {layout.num_images_padded}')
print(f'rasters    : {layout.num_rasters}')
print(f'gaps inside rasters : {layout.missing_image_index.size} at {layout.missing_image_index}')
print(f'short rasters       : {layout.raster_missing_end_index.size}, missing {layout.num_missing_images}')
print(f'total frames inserted: {layout.inserted.sum()}')


#### Reshape to create raster dimension


In [ ]:
sg_1394_dspk_reshape = layout.to_rasters(layout.pad(sg_1394_dspk))
sg_1403_dspk_reshape = layout.to_rasters(layout.pad(sg_1403_dspk))

print(sg_1394_dspk_reshape.shape)   # (raster, image, y, x)


In [ ]:
plt.figure()
plt.imshow(np.nanmean(sg_1394_dspk_reshape[0], axis=(-1)).T, aspect=1/6)
plt.colorbar()
plt.gca().invert_yaxis()
plt.title('Raster 0 before rolling trimmed mean')

#### Apply rolling trimmed mean along raster dimension
Using 'reflect' at the boundaries.

In [ ]:
%%time
# ROY'S NEW TRIMMED MEAN FILTER

# Trimmed mean applied to individual Si IV 1394 rasters:

average_1394 = np.empty_like(sg_1394_dspk_reshape)
for i in range(average_1394.shape[0]):
    average_1394[i] = ndfilters.trimmed_mean_filter(
        array=sg_1394_dspk_reshape[i],
        size=(32,1,1),
        where=np.isfinite(sg_1394_dspk_reshape[i]),
        proportion=0.35
    )

del sg_1394_dspk_reshape

In [ ]:
plt.figure()
plt.imshow(np.nanmean(average_1394[0], axis=(-1)).T, aspect=1/6)
plt.colorbar()
plt.gca().invert_yaxis()
plt.title('Raster 0 after rolling trimmed mean')

In [ ]:
%%time
# ROY'S NEW TRIMMED MEAN FILTER

# Trimmed mean applied to individual Si IV 1403 rasters:

average_1403 = np.empty_like(sg_1403_dspk_reshape)
for i in range(average_1403.shape[0]):
    average_1403[i] = ndfilters.trimmed_mean_filter(
        array=sg_1403_dspk_reshape[i],
        size=(32, 1, 1),
        where=np.isfinite(sg_1403_dspk_reshape[i]),
        proportion=0.35
    )

del sg_1403_dspk_reshape

#### Reshape back and remove the inserted frames

`from_rasters` drops the raster axis; `unpad` then removes exactly the frames
`pad` inserted, restoring the original image ordering and count.


In [ ]:
average_1394 = layout.unpad(layout.from_rasters(average_1394))
average_1403 = layout.unpad(layout.from_rasters(average_1403))

# sg_*_dspk were never padded in place, so they are already the original length
assert average_1394.shape == sg_1394_dspk.shape
assert average_1403.shape == sg_1403_dspk.shape


#### Reapply NaN masks

In [ ]:
average_1394[nan_mask_1394] = np.nan
average_1403[nan_mask_1403] = np.nan

#### Plots

In [ ]:
# For plotting purposes, take the mean of each of the trimmed mean arrays, so we have representative images for both.
average_1394_mean_image = np.nanmean(average_1394, axis=0)
average_1403_mean_image = np.nanmean(average_1403, axis=0)

In [ ]:
plot_lines_sidebyside(
    average_1394_mean_image,
    average_1403_mean_image,
    'Trimmed mean',
    size=(7,12),
    # percentile_min=.1,
    # percentile_max=99.9,
    # exp_max=35,
    # exp_min=3
);
 # .savefig('trimmed_mean_mar_2024.png',dpi=300, transparent=True)

#### Save rolling trimmed mean array

In [ ]:
with open(path / 'level_11_fpr_despiked_rtm_1394.pickle', 'wb') as fh:
    pickle.dump(average_1394, fh)

with open(path / 'level_11_fpr_despiked_rtm_1403.pickle', 'wb') as fh:
    pickle.dump(average_1403, fh)